# Korean Game Companies: Financial Health in 2025–2026

This notebook compares eight Korean listed game publishers using consolidated financial statements from the Financial Supervisory Service's Open DART API.

**Questions**

1. Which companies led FY2025 in scale and operating profitability?
2. How did Q1 2026 revenue and operating margins change year over year?
3. How do leverage and cash buffers differ across the peer group?
4. Which companies occupy the strongest growth–profitability quadrant?

All monetary values are KRW. This is descriptive analysis, not investment advice.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.titleweight": "bold"})

candidates = [
    Path("/kaggle/input/korean-game-companies-open-dart-2025-2026"),
    Path("../dataset"),
    Path("kaggle/dataset"),
]
DATA_DIR = next((path for path in candidates if (path / "financial_summary.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate the Kaggle dataset directory")

summary = pd.read_csv(DATA_DIR / "financial_summary.csv", dtype={"corp_code": str, "stock_code": str})
highlights = pd.read_csv(DATA_DIR / "financial_highlights_long.csv", dtype=str)
disclosures = pd.read_csv(DATA_DIR / "disclosures_2026.csv", dtype=str)
print(f"Data directory: {DATA_DIR}")
print(f"Companies: {len(summary)} | Highlight rows: {len(highlights):,} | 2026 filings: {len(disclosures):,}")
summary[["company_name", "market", "stock_code"]]

Data directory: ../dataset
Companies: 8 | Highlight rows: 256 | 2026 filings: 378


,company_name,market,stock_code
0,Com2uS,KOSDAQ,078340
1,Devsisters,KOSDAQ,194480
2,KRAFTON,KOSPI,259960
3,Kakao Games,KOSDAQ,293490
4,NC,KOSPI,036570
5,Netmarble,KOSPI,251270
6,Pearl Abyss,KOSDAQ,263750
7,WEMADE,KOSDAQ,112040


## Method and comparability

- Consolidated statements (`CFS`) are used for all eight companies.
- FY2025 values come from annual reports (`11011`).
- Q1 comparisons use first-quarter reports (`11013`) and their disclosed Q1 2025 comparatives.
- Annual and quarterly absolute amounts are never compared directly.
- Revenue growth uses `(Q1 2026 − Q1 2025) / Q1 2025`.
- A percentage change in operating profit can be misleading around zero or when its sign changes, so the notebook emphasizes margin and a turnaround/loss flag instead.
- Corrected filings are represented by the receipt numbers returned by the financial-statement API.

In [2]:
money_cols = [column for column in summary if column.endswith("_krw")]
assert summary["company_id"].nunique() == 8
assert summary[money_cols].notna().all().all()
assert (summary["currency"] == "KRW").all() if "currency" in summary else True

display_cols = [
    "company_name", "fy2025_revenue_krw", "fy2025_operating_margin_pct",
    "q1_revenue_yoy_pct", "q1_2026_operating_margin_pct", "q1_profit_direction",
    "q1_2026_debt_to_equity_pct", "q1_2026_cash_to_assets_pct",
]
summary[display_cols].sort_values("fy2025_revenue_krw", ascending=False).style.format({
    "fy2025_revenue_krw": "{:,.0f}",
    "fy2025_operating_margin_pct": "{:.1f}%",
    "q1_revenue_yoy_pct": "{:.1f}%",
    "q1_2026_operating_margin_pct": "{:.1f}%",
    "q1_2026_debt_to_equity_pct": "{:.1f}%",
    "q1_2026_cash_to_assets_pct": "{:.1f}%",
})

,company_name,fy2025_revenue_krw,fy2025_operating_margin_pct,q1_revenue_yoy_pct,q1_2026_operating_margin_pct,q1_profit_direction,q1_2026_debt_to_equity_pct,q1_2026_cash_to_assets_pct
2,KRAFTON,"3,326,553,694,966",31.7%,56.9%,40.9%,Improved,31.8%,6.9%
5,Netmarble,"2,835,074,411,896",12.4%,4.5%,8.1%,Improved,43.2%,9.1%
4,NC,"1,506,925,065,850",1.1%,54.7%,20.3%,Improved,29.4%,23.0%
0,Com2uS,"696,423,825,349",0.4%,-13.9%,3.5%,Improved,48.7%,10.0%
7,WEMADE,"614,039,504,365",1.7%,8.1%,5.5%,Turned profitable,121.4%,20.2%
3,Kakao Games,"465,018,566,528",-8.5%,-32.5%,-30.7%,Deteriorated,135.0%,20.6%
6,Pearl Abyss,"365,564,943,738",-4.1%,419.6%,64.6%,Improved,47.0%,10.3%
1,Devsisters,"295,578,347,074",2.1%,-34.4%,-29.7%,Turned to loss,89.3%,3.1%


## 1. FY2025 scale and profitability

In [3]:
ordered = summary.sort_values("fy2025_revenue_krw")
palette = dict(zip(sorted(summary.company_name), sns.color_palette("husl", len(summary))))
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True, gridspec_kw={"width_ratios": [1.4, 1]})
axes[0].barh(ordered.company_name, ordered.fy2025_revenue_krw / 1e12, color=[palette[x] for x in ordered.company_name])
axes[0].set(title="FY2025 Revenue", xlabel="KRW trillion", ylabel="")
axes[1].barh(ordered.company_name, ordered.fy2025_operating_margin_pct, color=[palette[x] for x in ordered.company_name])
axes[1].axvline(0, color="#333333", linewidth=1)
axes[1].set(title="FY2025 Operating Margin", xlabel="Percent", ylabel="")
axes[1].tick_params(axis="y", labelleft=False)
fig.suptitle("Korean Game Companies: Scale and Profitability", fontsize=20, fontweight="bold")
fig.subplots_adjust(wspace=.12)
plt.tight_layout()
plt.show()

/var/folders/ll/tcqsqfxd0_vc9n34tn2m7bwm0000gn/T/ipykernel_5765/3111213487.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Q1 2026 revenue growth

In [4]:
ordered = summary.sort_values("q1_revenue_yoy_pct")
colors = ["#2A9D8F" if value >= 0 else "#E76F51" for value in ordered.q1_revenue_yoy_pct]
plt.figure(figsize=(12, 7))
plt.barh(ordered.company_name, ordered.q1_revenue_yoy_pct, color=colors)
plt.axvline(0, color="#333333", linewidth=1)
plt.title("Q1 2026 Revenue Growth vs Q1 2025")
plt.xlabel("Year-over-year growth (%)")
plt.ylabel("")
plt.show()

/var/folders/ll/tcqsqfxd0_vc9n34tn2m7bwm0000gn/T/ipykernel_5765/2999753267.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Operating-margin movement

In [5]:
ordered = summary.sort_values("q1_2026_operating_margin_pct")
positions = np.arange(len(ordered)); width = 0.38
plt.figure(figsize=(14, 8))
plt.barh(positions - width/2, ordered.q1_2025_operating_margin_pct, height=width, label="Q1 2025", color="#A8DADC")
plt.barh(positions + width/2, ordered.q1_2026_operating_margin_pct, height=width, label="Q1 2026", color="#457B9D")
plt.yticks(positions, ordered.company_name)
plt.axvline(0, color="#333333", linewidth=1)
plt.title("Operating Margin: Q1 2025 vs Q1 2026")
plt.xlabel("Operating margin (%)"); plt.ylabel(""); plt.legend(frameon=False)
plt.show()
summary[["company_name", "q1_profit_direction"]].sort_values("company_name")

/var/folders/ll/tcqsqfxd0_vc9n34tn2m7bwm0000gn/T/ipykernel_5765/3175562301.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,company_name,q1_profit_direction
0,Com2uS,Improved
1,Devsisters,Turned to loss
2,KRAFTON,Improved
3,Kakao Games,Deteriorated
4,NC,Improved
5,Netmarble,Improved
6,Pearl Abyss,Improved
7,WEMADE,Turned profitable


## 4. Balance-sheet position

In [6]:
plt.figure(figsize=(12, 8))
sizes = 250 + summary.q1_2026_assets_krw / summary.q1_2026_assets_krw.max() * 1500
offsets = {"Pearl Abyss": (8,17), "Netmarble": (8,12), "Com2uS": (12,-22), "Kakao Games": (-98,18), "WEMADE": (10,-14)}
for idx, row in summary.iterrows():
    plt.scatter(row.q1_2026_debt_to_equity_pct, row.q1_2026_cash_to_assets_pct, s=sizes.loc[idx], color=palette[row.company_name], alpha=.8, edgecolor="white")
    plt.annotate(row.company_name, (row.q1_2026_debt_to_equity_pct, row.q1_2026_cash_to_assets_pct), xytext=offsets.get(row.company_name, (7,5)), textcoords="offset points", fontsize=10)
plt.title("Balance-Sheet Position at Q1 2026")
plt.xlabel("Debt-to-equity (%)"); plt.ylabel("Cash and equivalents / assets (%)")
plt.show()

/var/folders/ll/tcqsqfxd0_vc9n34tn2m7bwm0000gn/T/ipykernel_5765/623701414.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Growth–profitability map

In [7]:
plt.figure(figsize=(13, 8))
sizes = 250 + summary.q1_2026_revenue_krw / summary.q1_2026_revenue_krw.max() * 1500
plt.axhline(0, color="#777", linewidth=1); plt.axvline(0, color="#777", linewidth=1)
offsets = {"Pearl Abyss": (-78,10), "Netmarble": (8,14), "Com2uS": (-32,9), "WEMADE": (12,-28), "Devsisters": (8,14), "Kakao Games": (8,-18)}
for idx, row in summary.iterrows():
    plt.scatter(row.q1_revenue_yoy_pct, row.q1_2026_operating_margin_pct, s=sizes.loc[idx], color=palette[row.company_name], alpha=.82, edgecolor="white")
    plt.annotate(row.company_name, (row.q1_revenue_yoy_pct, row.q1_2026_operating_margin_pct), xytext=offsets.get(row.company_name, (7,5)), textcoords="offset points", fontsize=10)
plt.title("Q1 2026 Growth–Profitability Map")
plt.xlabel("Revenue growth YoY (%)"); plt.ylabel("Operating margin (%)")
plt.show()

/var/folders/ll/tcqsqfxd0_vc9n34tn2m7bwm0000gn/T/ipykernel_5765/2915186606.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Data-driven takeaways

In [8]:
largest = summary.loc[summary.fy2025_revenue_krw.idxmax()]
best_margin = summary.loc[summary.fy2025_operating_margin_pct.idxmax()]
fastest = summary.loc[summary.q1_revenue_yoy_pct.idxmax()]
cashiest = summary.loc[summary.q1_2026_cash_to_assets_pct.idxmax()]
print(f"• Largest FY2025 revenue: {largest.company_name} (KRW {largest.fy2025_revenue_krw/1e12:.2f}tn)")
print(f"• Highest FY2025 operating margin: {best_margin.company_name} ({best_margin.fy2025_operating_margin_pct:.1f}%)")
print(f"• Fastest Q1 revenue growth: {fastest.company_name} ({fastest.q1_revenue_yoy_pct:.1f}% YoY)")
print(f"• Highest Q1 cash/assets ratio: {cashiest.company_name} ({cashiest.q1_2026_cash_to_assets_pct:.1f}%)")
print("• Operating-profit direction:", summary.q1_profit_direction.value_counts().to_dict())

• Largest FY2025 revenue: KRAFTON (KRW 3.33tn)
• Highest FY2025 operating margin: KRAFTON (31.7%)
• Fastest Q1 revenue growth: Pearl Abyss (419.6% YoY)
• Highest Q1 cash/assets ratio: NC (23.0%)
• Operating-profit direction: {'Improved': 5, 'Turned to loss': 1, 'Deteriorated': 1, 'Turned profitable': 1}


## Limitations

- This is a small peer set of eight listed Korean game publishers, not the entire industry.
- Accounting classifications and consolidation scopes can differ by company.
- Q1 results can be seasonal and should not be annualized mechanically.
- Extreme growth rates can reflect a low comparison base, acquisitions, disposals, or major product launches; the filings should be read before assigning causality.
- Open DART republishes filer-submitted data and does not guarantee its accuracy or completeness.

**Source:** Financial Supervisory Service, Open DART. Analysis date: 2026-07-26. Not investment advice.